# Recurrent Neural Networks

Pichhle module mein humne text ke rich semantic representations aur embeddings ke upar ek simple linear classifier use kiya. Yeh architecture sentence mein words ka overall meaning capture karta hai, lekin yeh words ke **order** ko dhyan mein nahi rakhta, kyunki embeddings par kiya gaya aggregation operation original text se yeh information hata deta hai. Kyunki yeh models words ka order model nahi kar sakte, isliye woh text generation ya question answering jaise zyada complex ya ambiguous tasks solve nahi kar sakte.

Text sequence ka meaning capture karne ke liye, humein ek aur neural network architecture use karna hoga, jise **Recurrent Neural Network** ya RNN kehte hain. RNN mein hum apna sentence network ke through ek baar mein ek token pass karte hain, aur network kuch **state** produce karta hai, jise hum agle token ke saath network mein wapas pass kar dete hain.

Input sequence $X_0,\dots,X_n$ ke liye, RNN neural network blocks ki ek sequence banata hai aur is sequence ko back propagation se end-to-end train karta hai. Har network block $(X_i,S_i)$ ki ek pair ko input ke roop mein leta hai aur result mein $S_{i+1}$ produce karta hai. Final state $S_n$ ya output $X_n$ ko result produce karne ke liye ek linear classifier mein bheja jaata hai. Saare network blocks same weights share karte hain aur ek back propagation pass se end-to-end train hote hain.

Kyunki state vectors $S_0,\dots,S_n$ network ke through pass hote hain, yeh words ke beech sequential dependencies seekh sakta hai. Jaise, jab sequence mein kahin *not* word aata hai, toh yeh state vector ke kuch elements ko negate karna seekh sakta hai, jisse negation produce hota hai.

> Kyunki figure mein saare RNN blocks ke weights shared hote hain, isliye wahi figure ek single block (right side) ke roop mein bhi dikhai ja sakti hai, jisme ek recurrent feedback loop hota hai, jo network ki output state ko wapas input mein pass karta hai.

Chalo dekhte hain ki recurrent neural networks hamare news dataset ko classify karne mein kaise madad kar sakte hain.

In [ ]:
import torch
import torchtext
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)

## Simple RNN classifier

Simple RNN ke case mein, har recurrent unit ek simple linear network hota hai, jo combined input vector aur state vector leta hai, aur ek naya state vector produce karta hai. PyTorch is unit ko `RNNCell` class se represent karta hai, aur aise cells ke network ko `RNN` layer ke roop mein.

RNN classifier define karne ke liye, hum pehle embedding layer lagayenge taaki input vocabulary ki dimensionality kam ho jaye, aur phir uske upar RNN layer rakhenge:

In [ ]:
class RNNClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_class):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.rnn = torch.nn.RNN(embed_dim,hidden_dim,batch_first=True)
        self.fc = torch.nn.Linear(hidden_dim, num_class)

    def forward(self, x):
        batch_size = x.size(0)
        x = self.embedding(x)
        x,h = self.rnn(x)
        return self.fc(x.mean(dim=1))

> **Note:** Yahan hum simplicity ke liye ek untrained embedding layer use kar rahe hain, lekin aur behtar results ke liye hum Word2Vec ya GloVe embeddings ke saath pre-trained embedding layer use kar sakte hain, jaisa pichhli unit mein bataya gaya tha. Behtar samajh ke liye, aap is code ko pre-trained embeddings ke saath kaam karne ke liye adapt kar sakte hain.

Hamare case mein, hum padded data loader use karenge, jisse har batch mein same length wale padded sequences honge. RNN layer embedding tensors ki sequence legi aur do outputs produce karegi:
* $x$ har step par RNN cell outputs ki sequence hai
* $h$ sequence ke last element ke liye final hidden state hai

Uske baad hum ek fully-connected linear classifier lagayenge taaki classes ki number mil sake.

> **Note:** RNNs ko train karna kaafi mushkil hota hai, kyunki jab RNN cells ko sequence length ke saath unroll kiya jaata hai, toh back propagation mein shamil layers ki ginti bahut badi ho jaati hai. Isliye humein chhota learning rate chunna padta hai aur achhe results paane ke liye network ko bade dataset par train karna padta hai. Isme kaafi time lag sakta hai, isliye GPU use karna behtar hota hai.

In [ ]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)
net = RNNClassifier(vocab_size,64,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=0.001)

## Long Short Term Memory (LSTM)

Classical RNNs ki main problems mein se ek hai **vanishing gradients** problem. Kyunki RNNs ko ek hi back-propagation pass mein end-to-end train kiya jaata hai, yeh network ki shuruaati layers tak error propagate karne mein mushkil mehsoos karta hai, aur isi wajah se network door-door ke tokens ke beech relationships seekh nahi paata. Is problem se bachne ke tareekon mein se ek hai **explicit state management** ko implement karna, jo **gates** use karke kiya jaata hai. Is tarah ke do sabse mashhoor architectures hain: **Long Short Term Memory** (LSTM) aur **Gated Relay Unit** (GRU).

![Long Short Term Memory cell ka example dikhane wali image](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

LSTM network RNN ki tarah hi organize hota hai, lekin isme do states hoti hain jo layer se layer pass hoti hain: actual state $c$, aur hidden vector $h$. Har unit par, hidden vector $h_i$ ko input $x_i$ ke saath combine kiya jaata hai, aur yeh **gates** ke through control karte hain ki state $c$ par kya asar padega. Har gate ek neural network hota hai jisme sigmoid activation hoti hai (output $[0,1]$ ki range mein), jise state vector se multiply karne par bitwise mask ki tarah socha ja sakta hai. Yeh gates hote hain (upar di gayi picture mein left se right):
* **forget gate** hidden vector leta hai aur decide karta hai ki vector $c$ ke kaun se components ko bhoolna hai aur kaun se pass karne hain.
* **input gate** input aur hidden vector se kuch information leta hai aur use state mein daalta hai.
* **output gate** state ko $\tanh$ activation wali ek linear layer se transform karta hai, phir hidden vector $h_i$ use karke uske kuch components chunta hai taaki nayi state $c_{i+1}$ produce ho.

State $c$ ke components ko aise flags ki tarah socha ja sakta hai jinhe on aur off kiya ja sakta hai. Jaise, jab hum sequence mein naam *Alice* dekhte hain, toh hum maan sakte hain ki yeh ek female character ko refer karta hai, aur state mein flag raise karte hain ki sentence mein humare paas ek feminine noun hai. Jab aage hum phrase *and Tom* dekhte hain, toh hum flag raise karte hain ki humare paas ek plural noun hai. Is tarah state ko manipulate karke hum sentence ke parts ki grammatical properties ko track kar sakte hain.

> **Note**: LSTM ke internal structure ko samajhne ke liye ek behtareen resource hai Christopher Olah ka yeh shaandaar article [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/).

Haalaanki LSTM cell ka internal structure complex lag sakta hai, PyTorch is implementation ko `LSTMCell` class ke andar chhupa deta hai, aur poori LSTM layer ko represent karne ke liye `LSTM` object deta hai. Is tarah, LSTM classifier ka implementation upar dekhe gaye simple RNN jaisa hi hoga:

In [ ]:
class LSTMClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_class):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.embedding.weight.data = torch.randn_like(self.embedding.weight.data)-0.5
        self.rnn = torch.nn.LSTM(embed_dim,hidden_dim,batch_first=True)
        self.fc = torch.nn.Linear(hidden_dim, num_class)

    def forward(self, x):
        batch_size = x.size(0)
        x = self.embedding(x)
        x,(h,c) = self.rnn(x)
        return self.fc(h[-1])

In [ ]:
net = LSTMClassifier(vocab_size,64,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=0.001)

## Packed sequences

Hamare example mein, humein minibatch ki saari sequences ko zero vector se pad karna pada. Haalaanki isse thodi memory waste hoti hai, RNNs ke saath zyada important baat yeh hai ki padded input items ke liye extra RNN cells ban jaate hain, jo training mein hissa lete hain, lekin koi important input information nahi le jaate. Behtar hoga ki RNN ko sirf actual sequence size ke hisaab se train kiya jaye.

Isko karne ke liye, PyTorch mein padded sequence storage ka ek special format introduce kiya gaya hai. Maan lijiye humare paas aisa input padded minibatch hai:
```
[[1,2,3,4,5],
 [6,7,8,0,0],
 [9,0,0,0,0]]
```
Yahan 0 padded values ko represent karta hai, aur input sequences ki actual lengths ka vector `[5,3,1]` hai.

Padded sequence ke saath RNN ko effectively train karne ke liye, hum chahte hain ki RNN cells ke pehle group ki training bade minibatch (`[1,6,9]`) ke saath shuru ho, lekin phir teesri sequence ki processing khatam ho jaye, aur chhote minibatches (`[2,7]`, `[3,8]`) ke saath training jaari rahe, aur isi tarah aage. Is tarah, packed sequence ko ek vector ke roop mein represent kiya jaata hai - hamare case mein `[1,6,9,2,7,3,8,4,5]`, aur lengths ka vector (`[5,3,1]`), jisse hum original padded minibatch ko aasaani se reconstruct kar sakte hain.

Packed sequence banane ke liye, hum `torch.nn.utils.rnn.pack_padded_sequence` function use kar sakte hain. Saari recurrent layers, jaise RNN, LSTM aur GRU, input ke roop mein packed sequences ko support karti hain, aur packed output produce karti hain, jise `torch.nn.utils.rnn.pad_packed_sequence` se decode kiya ja sakta hai.

Packed sequence banane ke liye, humein network ko lengths ka vector pass karna hoga, aur isliye humein minibatches taiyaar karne ke liye ek alag function chahiye hoga:

In [ ]:
def pad_length(b):
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # compute max length of a sequence in this minibatch and length sequence itself
    len_seq = list(map(len,v))
    l = max(len_seq)
    return ( # tuple of three tensors - labels, padded features, length sequence
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v]),
        torch.tensor(len_seq)
    )

train_loader_len = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=pad_length, shuffle=True)

Actual network `LSTMClassifier` jaisa hi hoga, lekin `forward` pass mein padded minibatch aur sequence lengths ka vector, dono milenge. Embedding calculate karne ke baad, hum packed sequence calculate karte hain, use LSTM layer mein pass karte hain, aur phir result ko wapas unpack karte hain.

> **Note**: Hum actually unpacked result `x` use nahi karte, kyunki hum aage ki calculations mein hidden layer ka output use karte hain. Isliye, hum is code se unpacking ko poori tarah hata sakte hain. Ise yahan rakhne ka reason yeh hai ki aap is code ko aasaani se modify kar sakein, agar aapko aage ki calculations mein network output use karna pade.

In [ ]:
class LSTMPackClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_class):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.embedding.weight.data = torch.randn_like(self.embedding.weight.data)-0.5
        self.rnn = torch.nn.LSTM(embed_dim,hidden_dim,batch_first=True)
        self.fc = torch.nn.Linear(hidden_dim, num_class)

    def forward(self, x, lengths):
        batch_size = x.size(0)
        x = self.embedding(x)
        pad_x = torch.nn.utils.rnn.pack_padded_sequence(x,lengths,batch_first=True,enforce_sorted=False)
        pad_x,(h,c) = self.rnn(pad_x)
        x, _ = torch.nn.utils.rnn.pad_packed_sequence(pad_x,batch_first=True)
        return self.fc(h[-1])

In [ ]:
net = LSTMPackClassifier(vocab_size,64,32,len(classes)).to(device)
train_epoch_emb(net,train_loader_len, lr=0.001,use_pack_sequence=True)


> **Note:** Aapne dhyan diya hoga ki hum training function ko `use_pack_sequence` parameter pass karte hain. Abhi, `pack_padded_sequence` function ko lengths sequence tensor ka CPU device par hona zaroori hai, aur isliye training function ko training ke time lengths sequence data ko GPU par transfer karne se bachna padta hai. Aap [`torchnlp.py`](../../../../../lessons/5-NLP/16-RNN/torchnlp.py) file mein `train_emb` function ka implementation dekh sakte hain.

## Bidirectional aur multilayer RNNs

Hamare examples mein, saare recurrent networks ek direction mein kaam karte the, sequence ki shuruaat se end tak. Yeh natural lagta hai, kyunki yeh waise hi hai jaise hum padhte hain aur speech sunte hain. Lekin, kai practical cases mein humein input sequence ka random access milta hai, isliye dono directions mein recurrent computation chalana samajhdari ho sakti hai. Aise networks ko **bidirectional** RNNs kehte hain, aur inhe RNN/LSTM/GRU constructor mein `bidirectional=True` parameter pass karke banaya ja sakta hai.

Bidirectional network ke saath kaam karte waqt, humein do hidden state vectors chahiye honge, har direction ke liye ek. PyTorch in vectors ko ek bade size ke vector ke roop mein encode karta hai, jo kaafi convenient hai, kyunki aam taur par aap resulting hidden state ko fully connected linear layer mein pass karte hain, aur aapko bas layer banate waqt size ki is badhotri ko dhyan mein rakhna hota hai.

Recurrent network, chahe woh one-directional ho ya bidirectional, sequence ke andar kuch patterns capture karta hai aur unhe state vector mein store kar sakta hai ya output mein pass kar sakta hai. Jaise convolutional networks ke saath hota hai, hum pehli layer dwara nikaale gaye low-level patterns se high-level patterns capture karne ke liye pehli layer ke upar ek aur recurrent layer bana sakte hain. Isse hum **multilayer RNN** ke concept tak pahunchte hain, jisme do ya zyada recurrent networks hote hain, jahan pichhli layer ka output agli layer ko input ke roop mein pass kiya jaata hai.

![Multilayer long-short-term-memory RNN dikhane wali image](../../../../../lessons/5-NLP/16-RNN/images/multi-layer-lstm.jpg)

*Yeh picture [is shaandaar post](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) se li gayi hai, jise Fernando Lopez ne likha hai.*

PyTorch aise networks banana aasaan bana deta hai, kyunki aapko bas RNN/LSTM/GRU constructor mein `num_layers` parameter pass karna hota hai, jisse recurrence ki kai layers automatically ban jaati hain. Iska matlab yeh bhi hoga ki hidden/state vector ka size proportionally badhega, aur recurrent layer ke output ko handle karte waqt aapko ise dhyan mein rakhna hoga.

## Dusre tasks ke liye RNNs

Is unit mein humne dekha ki RNNs ko sequence classification ke liye use kiya ja sakta hai, lekin actually, woh kai aur tasks sambhal sakte hain, jaise text generation, machine translation, aur bhi bahut kuch. Hum in tasks par agli unit mein vichaar karenge.

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->